# Gemini Model Migration Guide for MathVerse Live Tutoring

**Objective**: Migrate MathVerse live tutoring backend from deprecated Gemini 2.0 Flash models to Gemini 3.1 Flash Lite (or 2.5 Flash Lite) for continued service with Google Cloud Vertex AI Live API.

## Migration Timeline
- **Deprecation Notice**: June 1, 2026 (Google Cloud official discontinuation date)
- **Affected Projects**: 
  - `effective-fire-414301` (primary Vertex AI project)
  - `mathverse-live-ai` (production tutoring service)

## Recommended Migration Paths
1. **Primary**: Gemini 3.1 Flash Lite (Generally Available soon)
2. **Alternative 1**: Gemini 2.5 Flash Lite (stable, currently tested)
3. **Alternative 2**: Gemma 4 (if audio support differs)

## What This Guide Covers
- Current model configuration and deprecation status
- Step-by-step migration from deprecated models
- Configuration updates in `config.py`
- Connection setup changes for new models
- Audio streaming parameter updates
- System prompt migration
- Testing procedures with new models
- Production deployment checklist

## Section 1: Current Configuration and Deprecation Status

### Current Model in Use
The `backend/app/core/config.py` currently specifies:

```python
GEMINI_LIVE_MODEL = os.getenv(
    "GEMINI_LIVE_MODEL",
    "gemini-2.5-flash-native-audio-preview-12-2025",
)
```

**Status**: Using a preview model with native audio support. This is newer than the deprecated `gemini-2.0-flash-exp`, but still requires migration to GA models per Google Cloud announcement.

### Why Migration is Required
1. **Official Deprecation**: Google Cloud discontinuing Gemini 2.0 Flash and Flash Lite on June 1, 2026
2. **API Changes**: Older models may lose support for `bidiGenerateContent` endpoint (Live API)
3. **Production Stability**: GA models provide guaranteed SLA and support
4. **Feature Parity**: Newer models offer improved latency, quality, and contextual understanding

### Deprecated Models NOT to Use After June 1, 2026
- `gemini-2.0-flash-exp`
- `gemini-2.0-flash-lite-exp`
- `gemini-2.0-flash-001`
- Vertex AI versions of above

## Section 2: Step 1 - Update Configuration (config.py)

### Required Change

Edit `backend/app/core/config.py` to update the default Gemini Live model:

**BEFORE (Deprecated/Preview):**
```python
GEMINI_LIVE_MODEL = os.getenv(
    "GEMINI_LIVE_MODEL",
    "gemini-2.5-flash-native-audio-preview-12-2025",  # Preview with audio
)
```

**AFTER (GA Model - Recommended):**
```python
GEMINI_LIVE_MODEL = os.getenv(
    "GEMINI_LIVE_MODEL",
    "gemini-3.1-flash-lite",  # GA model with Live API support
)
```

### Alternative Configuration Options

If Gemini 3.1 Flash Lite is not yet available in your region, use these alternatives:

**Option A - Gemini 2.5 Flash Lite (Stable):**
```python
GEMINI_LIVE_MODEL = os.getenv(
    "GEMINI_LIVE_MODEL",
    "gemini-2.5-flash-lite",  # Stable alternative
)
```

**Option B - Using Environment Variables (Recommended for Production):**
Instead of changing code, set the environment variable:
```bash
export GEMINI_LIVE_MODEL=gemini-3.1-flash-lite
```

This allows different deployments to use different models without code changes.

### Verification

After update, verify the configuration loads correctly:
```python
from backend.app.core.config import GEMINI_LIVE_MODEL
print(f"Configured model: {GEMINI_LIVE_MODEL}")
# Expected output: Configured model: gemini-3.1-flash-lite
```

## Section 3: Step 2 - Verify Audio Transcription Configuration

### Important Fix Already Applied

The code in `backend/app/services/live_tutor_service.py` (lines 195-197) has already been corrected to remove unsupported `language_codes` parameter:

```python
# ✅ CORRECT - As currently implemented:
input_audio_transcription=live_types.AudioTranscriptionConfig(),
output_audio_transcription=live_types.AudioTranscriptionConfig(),

# ❌ INCORRECT - Do NOT use this pattern:
input_audio_transcription=live_types.AudioTranscriptionConfig(
    language_codes=[GEMINI_LIVE_INPUT_LANGUAGE, "hi-IN"]  # Not supported!
),
```

### Why This Change Was Made

1. **API Limitation**: Google Gemini Live API does not support `language_codes` parameter in `AudioTranscriptionConfig`
2. **Error Message**: Was causing "language_codes parameter is not supported in Gemini API" errors
3. **Correct Approach**: Use `speech_config.language_code` instead (which IS supported)

### Current Correct Implementation

```python
speech_config=live_types.SpeechConfig(
    language_code=GEMINI_LIVE_OUTPUT_LANGUAGE,  # ✅ This IS supported
    voice_config=live_types.VoiceConfig(
        prebuilt_voice_config=live_types.PrebuiltVoiceConfig(
            voice_name=GEMINI_LIVE_VOICE
        )
    ),
),
input_audio_transcription=live_types.AudioTranscriptionConfig(),  # ✅ Empty config
output_audio_transcription=live_types.AudioTranscriptionConfig(),  # ✅ Empty config
```

### No Changes Needed for Gemini 3.1 Migration

This configuration pattern is compatible with Gemini 3.1 Flash Lite. No modifications required.

## Section 4: Step 3 - Verify System Prompt and Agent Configuration

### Teacher Agent System Prompt

The TutorAgent in `backend/app/agents/teacher_agent.py` has been updated with a balanced teaching approach that is compatible with all Gemini models (2.5 and 3.1+).

**Current SYSTEM_PROMPT Characteristics:**
- ✅ Allows natural explanations and concept blocks
- ✅ Balances Socratic questioning with direct instruction
- ✅ Includes "Move On Rule" for stuck students
- ✅ No model-specific constraints

### System Prompt Components (No Changes Required)

The system prompt is constructed dynamically with:
1. **Base instruction** from `TutorAgent.SYSTEM_PROMPT`
2. **Live audio rules** (keep turns short, handle interruptions)
3. **Chapter focus** and curriculum grounding
4. **Memory context** from previous turns

### Compatibility Check

```python
# Verify system prompt loads correctly
from backend.app.agents.teacher_agent import TutorAgent

tutor = TutorAgent()
print("System prompt length:", len(tutor.SYSTEM_PROMPT))
print("First 200 chars:", tutor.SYSTEM_PROMPT[:200])
# Should print without errors for both Gemini 2.5 and 3.1
```

### No Migration Changes Needed Here

The system prompt is model-agnostic and works with both Gemini 2.5 and 3.1 Flash Lite.

## Section 5: Step 4 - Audio Configuration and Sample Rates

### Current Audio Configuration

The audio configuration in `backend/app/services/live_tutor_service.py` is already optimized and compatible with Gemini 3.1:

```python
# Audio input (student speaking)
LIVE_INPUT_SAMPLE_RATE = 16000  # Hz (standard for speech recognition)

# Audio output (tutor speaking)
LIVE_OUTPUT_SAMPLE_RATE = 24000  # Hz (Gemini Live API standard)
```

### Audio MIME Type Configuration

When sending audio to Gemini Live:
```python
mime_type=f"audio/pcm;rate={LIVE_INPUT_SAMPLE_RATE}"
# Results in: "audio/pcm;rate=16000"
```

This format is supported by both Gemini 2.5 and Gemini 3.1 Flash Lite.

### Voice Configuration

The voice configuration uses a named voice from Google's prebuilt voice catalog:

```python
speech_config=live_types.SpeechConfig(
    language_code=GEMINI_LIVE_OUTPUT_LANGUAGE,  # "en-IN" or "en-US"
    voice_config=live_types.VoiceConfig(
        prebuilt_voice_config=live_types.PrebuiltVoiceConfig(
            voice_name=GEMINI_LIVE_VOICE  # "Sulafat" or other prebuilt voices
        )
    ),
),
```

### Available Voices in Gemini 3.1

Gemini 3.1 Flash Lite supports the same prebuilt voice names as 2.5:
- `Aoede`, `Charon`, `Fenrir`, `Kore`, `Orion`, `Puck`, `Scarlett`, `Serena`, `Staccato`, `Stern`, `Sulafat`

**No changes required** - current voice configuration will work with Gemini 3.1.

## Section 6: Migration Checklist

### Pre-Migration Verification

- [ ] Review Google Cloud deprecation notice for your project
- [ ] Identify all files using `GEMINI_LIVE_MODEL` constant
- [ ] Check for any hardcoded model names (search for "gemini-2.0" in codebase)
- [ ] Verify API key and authentication credentials are valid
- [ ] Test connectivity to Google AI or Vertex AI endpoints

### Configuration Updates

- [ ] Update `backend/app/core/config.py`: Change `GEMINI_LIVE_MODEL` to `gemini-3.1-flash-lite`
- [ ] Set environment variable `GEMINI_LIVE_MODEL` for production deployment
- [ ] Verify no other files have hardcoded model names
- [ ] Check requirements.txt for compatible google-genai SDK version

### Code Verification

- [ ] Confirm `live_tutor_service.py` has empty `AudioTranscriptionConfig()` (lines 195-197)
- [ ] Verify `teacher_agent.py` SYSTEM_PROMPT loads correctly
- [ ] Check all imports from `google.genai` are still valid
- [ ] Run Python import tests to catch any compatibility issues

### Testing (Pre-Production)

- [ ] Test WebSocket connection to Gemini Live API with new model
- [ ] Verify bidirectional audio streaming (send 16kHz PCM, receive 24kHz)
- [ ] Test input transcription (student speech → text)
- [ ] Test output transcription (model speech → text)
- [ ] Test turn completion events and state management
- [ ] Verify session resumption works correctly
- [ ] Test multiple consecutive turns without disconnection

### Deployment Phases

#### Phase 1: Staging Environment
- [ ] Deploy to staging with Gemini 3.1 Flash Lite
- [ ] Run full integration test suite
- [ ] Perform load testing with concurrent connections
- [ ] Monitor logs for any API errors or incompatibilities
- [ ] Duration: 1-2 weeks of testing

#### Phase 2: Limited Production Rollout
- [ ] Deploy to 10-20% of production servers
- [ ] Monitor WebSocket connection stability
- [ ] Track response latency and quality metrics
- [ ] Gather feedback from real users
- [ ] Duration: 1 week monitoring

#### Phase 3: Full Production Rollout
- [ ] Deploy to remaining production servers
- [ ] Monitor system health metrics
- [ ] Keep rollback plan ready if issues arise
- [ ] Duration: Gradual rollout over 2-3 days

### Rollback Plan

If migration to Gemini 3.1 encounters issues:
1. Revert `config.py` to previous model
2. Set `GEMINI_LIVE_MODEL=gemini-2.5-flash-lite` (stable alternative)
3. Restart affected services
4. Monitor connection stability
5. Investigate root cause before re-attempting migration

In [ ]:
# Quick Test 1: Verify Configuration Loads

import os
import sys

# Add backend to path
sys.path.insert(0, '/path/to/mathverse/backend')

try:
    from app.core.config import GEMINI_LIVE_MODEL
    print(f"✅ Configuration loaded successfully")
    print(f"   Model: {GEMINI_LIVE_MODEL}")
    
    if "3.1" in GEMINI_LIVE_MODEL:
        print(f"   Status: ✅ Using recommended Gemini 3.1 Flash Lite")
    elif "2.5" in GEMINI_LIVE_MODEL:
        print(f"   Status: ✅ Using Gemini 2.5 (alternative)")
    elif "2.0" in GEMINI_LIVE_MODEL:
        print(f"   Status: ⚠️  WARNING: Still using deprecated Gemini 2.0!")
    else:
        print(f"   Status: ❓ Unknown model version")
        
except ImportError as e:
    print(f"❌ Failed to import config: {e}")
    print("   Make sure you've set PYTHONPATH correctly")

In [ ]:
# Quick Test 2: Check for Deprecated Model Names in Codebase

import os
import re
from pathlib import Path

backend_path = Path('/path/to/mathverse/backend')
deprecated_models = ['gemini-2.0', 'gemini-2.0-flash-exp', 'gemini-2.0-flash-lite-exp']

def find_deprecated_references():
    """Search for deprecated model names in Python files"""
    matches = []
    
    if not backend_path.exists():
        print("⚠️  Backend path not found. Update the path and try again.")
        return matches
    
    for py_file in backend_path.rglob('*.py'):
        with open(py_file, 'r') as f:
            content = f.read()
            for deprecated in deprecated_models:
                if deprecated in content:
                    # Find line numbers
                    for i, line in enumerate(content.split('\n'), 1):
                        if deprecated in line and not line.strip().startswith('#'):
                            matches.append({
                                'file': str(py_file.relative_to(backend_path)),
                                'line': i,
                                'content': line.strip()
                            })
    
    return matches

print("Searching for deprecated model names...")
deprecated_refs = find_deprecated_references()

if deprecated_refs:
    print(f"\n⚠️  Found {len(deprecated_refs)} references to deprecated models:\n")
    for ref in deprecated_refs:
        print(f"  {ref['file']}:{ref['line']}")
        print(f"    {ref['content'][:80]}...")
else:
    print("\n✅ No deprecated model names found in codebase!")

In [ ]:
# Quick Test 3: Verify Google Genai SDK Compatibility

try:
    import google.genai as genai
    from google.genai import types as live_types
    
    print("✅ google.genai SDK installed")
    print(f"   Version: {getattr(genai, '__version__', 'unknown')}")
    
    # Check for required types
    required_types = [
        'LiveConnectConfig',
        'AudioTranscriptionConfig',
        'SpeechConfig',
        'VoiceConfig',
        'PrebuiltVoiceConfig'
    ]
    
    missing_types = []
    for type_name in required_types:
        if not hasattr(live_types, type_name):
            missing_types.append(type_name)
    
    if missing_types:
        print(f"⚠️  Missing types: {missing_types}")
    else:
        print(f"✅ All required types available for Gemini Live API")
        
except ImportError as e:
    print(f"❌ google.genai not installed: {e}")
    print("   Install with: pip install google-genai")

## Summary: What to Do Next

### Immediate Actions (Before June 1, 2026)

1. **Update Configuration**
   - Edit `backend/app/core/config.py`
   - Change `GEMINI_LIVE_MODEL` default from preview model to `gemini-3.1-flash-lite`
   - Or set environment variable: `export GEMINI_LIVE_MODEL=gemini-3.1-flash-lite`

2. **Run Verification Tests**
   - Use the test cells in this notebook to verify configuration and dependencies
   - Search codebase for any hardcoded model names
   - Confirm SDK types are available

3. **Deployment to Staging**
   - Deploy updated config to staging environment
   - Run integration tests with full audio pipeline
   - Monitor WebSocket connection stability for 1-2 weeks

4. **Production Rollout**
   - Execute phased deployment (10% → 50% → 100%)
   - Have rollback plan ready
   - Monitor metrics continuously

### Files Modified in This Migration

| File | Change | Impact |
|------|--------|--------|
| `backend/app/core/config.py` | Update `GEMINI_LIVE_MODEL` | ✅ Primary change |
| `backend/app/services/live_tutor_service.py` | Already fixed - no changes needed | ✅ Verified correct |
| `backend/app/agents/teacher_agent.py` | Already updated - no changes needed | ✅ Compatible |
| `requirements.txt` | May need google-genai version bump | Check for compatibility |

### Troubleshooting Guide

**Problem**: WebSocket connection fails with new model
- **Solution**: Check `GEMINI_LIVE_MODEL` environment variable is set correctly
- **Fallback**: Use `gemini-2.5-flash-lite` until 3.1 is available in your region

**Problem**: Audio transcription not working
- **Solution**: Verify `AudioTranscriptionConfig()` is empty (no language_codes parameter)
- **Reference**: See Section 3 for correct implementation

**Problem**: Voice not sounding natural
- **Solution**: Try different prebuilt voice names (Aoede, Sulafat, Serena, etc.)
- **Note**: Voice availability may vary between models

**Problem**: Latency increased with new model
- **Solution**: Model performance improves with usage - allow 1-2 weeks for cache warmup
- **Alternative**: Contact Google Cloud support for model-specific optimization

### Support Resources

- Google Cloud Console: Monitor API quotas and logs
- Gemini API Documentation: https://ai.google.dev/docs
- Vertex AI Live API Docs: Check Google Cloud documentation
- MathVerse Team: Internal support channel

### Timeline Reminder

⏰ **Deadline**: June 1, 2026 - After this date, deprecated models will not be accessible
📅 **Current Status**: Plan migration now, test thoroughly, deploy by Q1 2026